# DISC5105 Week 2 – A1: Multimodal Data Fusion with Video and Insole Pressure

Complete the blanks (`____`) in the code. Each blank is preceded by a
`# TODO ... (one line):` comment explaining what to fill in.
All other code is supplied. Complete each task's blanks before running that cell.
Keep the existing quotation marks when filling the interpolation mode.
After finishing, restart the kernel, run all cells, and save the notebook with its outputs.


### Setup

Import the required libraries and define the supplied video, pressure-data, and processing settings. Use the provided filenames unless the instructor gives you a different data pair. The constants in this section define the frame and image size used throughout the assignment.

**Automatic data download.** If `S2.mp4` or `S2.txt` is not present in the notebook's working directory, the next code cell downloads it from the course GitHub dataset. Existing local files are used unchanged. Internet access is required only when a file is missing.


In [ ]:
from pathlib import Path

import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

np.set_printoptions(precision=3, suppress=True)

ROOT = Path(".")
VIDEO = ROOT / "S2.mp4"
PRESSURE = ROOT / "S2.txt"
FRAME_ID = 30
IMAGE_SIZE = 128

from urllib.request import urlretrieve

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/yantaolab/Autonomous-Construction-and-Robotics/"
    "main/Projects/Project%2006%20Construction%20safety%20and%20health/dataset"
)

def download_if_missing(path, url):
    if path.exists() and path.stat().st_size > 0:
        print(f"Using local file: {path}")
        return
    print(f"Downloading {path.name} from the course dataset...")
    urlretrieve(url, path)
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError(f"Download failed or produced an empty file: {path}")

download_if_missing(VIDEO, f"{DATA_BASE_URL}/S2.mp4")
download_if_missing(PRESSURE, f"{DATA_BASE_URL}/S2.txt")


## Read the supplied data

The provided loader reads `S2.txt` into timestamps (`times`) and left/right insole measurements (`left` and `right`). Each foot contributes 16 pressure sensors. Treat this cell as reference code: do not change the parser or the sensor-coordinate definition.


### Load pressure data

This supplied cell reads the tab-separated pressure file, separates the 16 left-foot and 16 right-foot sensors, and records their locations on a 13×13 grid. Run it unchanged and confirm the printed array shapes before moving on.


In [ ]:
def read_pressure_file(txt_path):
    """Read S2.txt and return timestamps plus left/right pressure sequences.

    Returns
    -------
    times : np.ndarray, shape (N,)
    left_pressures : np.ndarray, shape (N, 16)
    right_pressures : np.ndarray, shape (N, 16)
    """
    rows = []
    for line in Path(txt_path).read_text(encoding="utf-8", errors="replace").splitlines():
        if not line.strip() or line.startswith("#"):
            continue
        try:
            values = [float(v) if v.strip() else 0.0 for v in line.rstrip("\r\n").split("\t")]
        except ValueError:
            continue
        rows.append((values + [0.0] * 51)[:51])
    data = np.asarray(rows, dtype=np.float32)
    if data.ndim != 2 or data.shape[1] != 51:
        raise ValueError("Expected an N x 51 pressure matrix")
    return data[:, 0], data[:, 1:17], data[:, 26:42]

LEFT_SENSOR_POSITIONS = [
    (0, 3), (1, 2), (1, 3), (1, 4),
    (2, 1), (2, 2), (2, 3), (2, 4),
    (4, 2), (4, 3), (6, 2), (6, 3),
    (9, 1), (9, 2), (10, 2), (10, 3),
]

times, left, right = read_pressure_file(PRESSURE)
print("times:", times.shape, "left:", left.shape, "right:", right.shape)

def print_pressure_evidence(pressure_map, left_pressures, right_pressures, matched_index):
    print()
    print("===== Pressure-map evidence =====")
    print("pressure_map[0] (13×13):")
    print(pressure_map[0])
    print("Sensor coordinates and values (row, col, value):")
    for sensor_id, (row, col) in enumerate(LEFT_SENSOR_POSITIONS):
        print(f"L{sensor_id+1:02d}: ({row:02d},{col:02d})={left_pressures[matched_index, sensor_id]:.3f} | "
              f"R{sensor_id+1:02d}: ({row:02d},{12-col:02d})={right_pressures[matched_index, sensor_id]:.3f}")


## Task 1 – Video–pressure time alignment and spatial mapping

**Goal.** Link one video frame to the physically corresponding insole-pressure observation, then represent both feet on a common 13×13 grid.

**Key idea.** Calculate the frame time (frame_id / fps) and match it with pressure samples using absolute timestamps. Keep the left foot sensor coordinates unchanged and mirror the right foot coordinates around the center column (right_col = 12 − left_col) to make the feet spatially comparable.

**Evidence to retain.** The output should show the frame time, matched pressure time, absolute timing error, the `(1, 13, 13)` pressure-map shape, and the complete matrix. The map should contain 32 placed sensor values (16 per foot).


### Read one video frame and construct the pressure map

Complete the three blanks marked TODO 1–3. The comments immediately before
the marked lines explain how to match the timestamps and place the sensor values.
Run the cell to print the alignment results and pressure map.


In [ ]:
cap = cv2.VideoCapture(str(VIDEO))
if not cap.isOpened():
    raise FileNotFoundError(VIDEO)
fps = float(cap.get(cv2.CAP_PROP_FPS) or 30.0)
cap.set(cv2.CAP_PROP_POS_FRAMES, FRAME_ID)
ok, frame = cap.read()
cap.release()
if not ok:
    raise RuntimeError(f"Could not read frame {FRAME_ID}")
print("fps:", fps, "frame shape (BGR):", frame.shape, "frame time:", FRAME_ID / fps)

def align_and_build_pressure_map(times, left_pressures, right_pressures, frame_id, fps):
    """Align already-read TXT pressure data to one video frame and build a 1x13x13 map.

    Requirements:
    1. frame_time = frame_id / fps.
    2. Find the TXT row with the smallest absolute time difference from frame_time.
    3. Create a zero matrix with shape=(1, 13, 13).
    4. Write the row's 16 left-foot values into LEFT_SENSOR_POSITIONS.
    5. Write the 16 right-foot values into mirrored positions: right column = 12 - left column.
    6. Return pressure_map and matched_time.

    Returns
    -------
    pressure_map : np.ndarray, shape (1, 13, 13), dtype float32
    matched_time : float
    """
    frame_time = frame_id / fps

    # TODO 1 (one line): Fill in the variable that stores the video frame time to find the nearest pressure timestamp.
    nearest_index = int(np.argmin(np.abs(times - ____)))

    pressure_map = np.zeros((1, 13, 13), dtype=np.float32)
    for sensor_id, (row, col) in enumerate(LEFT_SENSOR_POSITIONS):
        # TODO 2 (one line): Fill in the current sensor index to read its left-foot pressure value.
        pressure_map[0, row, col] = left_pressures[nearest_index, ____]
        # TODO 3 (one line): Fill in the mirrored column expression. The grid has column indices 0 to 12.
        pressure_map[0, ____, 12-col] = right_pressures[nearest_index, sensor_id]

    return pressure_map, float(times[nearest_index])

matched_index = int(np.argmin(np.abs(times - FRAME_ID / fps)))
pressure_map, matched_time = align_and_build_pressure_map(times, left, right, FRAME_ID, fps)
print()
print("===== Pressure-map output =====")
print(f"frame_time={FRAME_ID / fps:.6f}s")
print(f"matched_time={matched_time:.6f}s")
print(f"absolute_time_error={abs(matched_time - FRAME_ID / fps):.6f}s")
print("pressure_map shape:", pressure_map.shape)
print("pressure_map[0] (13×13):")
print(pressure_map[0])
print(f"matched_pressure_index={matched_index}")
print_pressure_evidence(pressure_map, left, right, matched_index)


: 

## Task 2 – Four-channel multimodal fusion

**Goal.** Convert the RGB video frame and its aligned pressure map into one tensor that a single neural network can consume.

**Key idea.** RGB provides three image channels; the pressure map is the fourth modality. Divide pressure values by a robust scale factor, resize the result to the image resolution with bilinear interpolation, and concatenate the channels in the order `[R, G, B, pressure]`. This preserves both visual and plantar-pressure information.

**Evidence to retain.** The fused tensor must have shape `(1, 4, H, W)`. Check the reported RGB and pressure ranges, the sampled `[R, G, B, P]` pixel, and the pressure preview to confirm that the fourth channel is finite and non-empty.


### Create the multimodal input tensor

Complete the three blanks marked TODO 4–6. The comments explain pressure
normalisation, bilinear upsampling, and concatenation along the channel dimension.
The fused tensor should have shape `(1, 4, 128, 128)`.


In [ ]:
def make_four_channel_input(frame_bgr, pressure_map, pressure_scale, image_size):
    """Create the fused tensor that is fed into HRNet.

    Requirements:
    1. Resize the BGR video frame to image_size x image_size and convert it to RGB.
    2. The RGB tensor should have shape=(1, 3, H, W) and values in [0, 1].
    3. Normalize pressure_map (1, 13, 13) by dividing it by pressure_scale.
    4. Use F.interpolate(..., mode="bilinear") to upsample it to (1, 1, H, W).
    5. Fuse with torch.cat(..., dim=1) and return shape=(1, 4, H, W).

    Returns
    -------
    fused : torch.Tensor, shape (1, 4, H, W)
    """
    frame = cv2.resize(frame_bgr, (image_size, image_size), interpolation=cv2.INTER_AREA)
    rgb = torch.from_numpy(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).copy()).float().permute(2, 0, 1) / 255.0
    pressure = torch.from_numpy(pressure_map).unsqueeze(0)

    # TODO 4 (one line): Fill in the supplied pressure-scale variable to normalise the pressure values.
    pressure = pressure / max(float(____), 1e-6)
    # TODO 5 (one line): Fill in the interpolation mode for bilinear upsampling. Keep the quotation marks.
    pressure = F.interpolate(pressure, size=(image_size, image_size), mode="____", align_corners=False)
    # TODO 6 (one line): Fill in the channel dimension for a tensor ordered as (batch, channels, height, width).
    fused = torch.cat([rgb.unsqueeze(0), pressure], dim=____)

    return fused

pressure_scale = float(np.percentile(np.concatenate([left.ravel(), right.ravel()]), 95))
fused = make_four_channel_input(frame, pressure_map, pressure_scale, IMAGE_SIZE)
pressure_channel = fused[:, 3:4]
print()
print("===== Four-channel fusion output =====")
print(f"pressure_scale={pressure_scale:.6f}")
print("fused shape:", tuple(fused.shape))
print("RGB range:", float(fused[:, :3].min()), "to", float(fused[:, :3].max()))
print("pressure-channel range:", float(pressure_channel.min()), "to", float(pressure_channel.max()))
print("one fused pixel [R,G,B,P]:", fused[0, :, 0, 0].detach().cpu().numpy())
print("pressure channel 5x5 sampled preview:")
print(pressure_channel[0, 0, ::max(1, IMAGE_SIZE // 4), ::max(1, IMAGE_SIZE // 4)].detach().cpu().numpy()[:5, :5])


## Task 3 – Simplified four-channel HRNet

**Goal.** Build a compact two-resolution network that combines fine spatial detail with broader context before predicting 17 keypoint heatmaps.

**Key idea.** The high-resolution branch retains positional detail for locating joints. A lower-resolution branch has a larger receptive field and can integrate wider visual and pressure context. Project its features to 32 channels, upsample them to the high-resolution size, and add the two branches before the prediction head.

**Evidence to retain.** The model printout and shape trace should follow the supplied channel/resolution contract and end with 17 heatmaps. The network is intentionally small and randomly initialised; structural correctness, not pose accuracy, is being assessed.


### Define the pose-estimation network

The residual block and network structure are supplied. Complete the five blanks
marked TODO 7–11 using the comments before each line and the shape hints in the docstring.
The next task prints the network and its tensor shapes.


In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False), nn.BatchNorm2d(channels),
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(x + self.layers(x))

class FourChannelHRNet(nn.Module):
    """Build a simplified teaching four-channel HRNet from these shape hints.

    input X:             (B,  4, 128, 128)   # RGB + pressure
    stem(X):             (B, 32,  32,  32)   # two stride=2 downsampling layers
    high(stem):          (B, 32,  32,  32)   # high-resolution branch
    down(high):          (B, 64,  16,  16)   # low-resolution branch
    low(down):           (B, 64,  16,  16)
    fuse(low):           (B, 32,  16,  16)   # use a 1x1 convolution to change channels
    upsample(fuse):      (B, 32,  32,  32)   # align spatial size with high
    high + upsample:     (B, 32,  32,  32)
    head(...):           (B, 17,  32,  32)   # 17 keypoint heatmaps

    When image_size is not 128, H/W follow the same 1/4 and 1/8 ratios.
    """
    def __init__(self, joints=17):
        super().__init__()
        self.stem = nn.Sequential(
            # TODO 7 (one line): Fill in the input channel count: three RGB channels plus one pressure channel.
            nn.Conv2d(____, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, stride=2, padding=1, bias=False), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
        )
        self.high = nn.Sequential(BasicBlock(32), BasicBlock(32))
        # TODO 8 (one line): Fill in the stride that reduces the spatial size from 32 x 32 to 16 x 16, strid 2.
        self.down = nn.Sequential(nn.Conv2d(32, 64, 3, stride=____, padding=1, bias=False), nn.BatchNorm2d(64), nn.ReLU(inplace=True))
        self.low = nn.Sequential(BasicBlock(64), BasicBlock(64))
        # TODO 9 (one line): Fill in the output channel count so the low-resolution features match the 32-channel high-resolution branch.
        self.fuse = nn.Sequential(nn.Conv2d(64, 32, 1, bias=False), nn.BatchNorm2d(____))
        self.head = nn.Conv2d(32, joints, 1)

    def forward(self, x):
        high = self.high(self.stem(x))
        low = self.low(self.down(high))
        low_up = F.interpolate(self.fuse(low), size=high.shape[-2:], mode="bilinear", align_corners=False)
        return self.head(high + low_up)


## Task 4 – Heatmap decoding and pipeline verification

**Goal.** Convert the network output into a readable 17×3 keypoint matrix and verify the complete video–pressure fusion pipeline.

**Key idea.** Each heatmap corresponds to one joint. Take its maximum activation as the predicted grid location, scale that location back to image coordinates, and report `[x, y, confidence]`. The automated checks validate shapes, alignment, mirroring, finiteness, and tensor logic; they do not evaluate pose-estimation accuracy because the teaching network uses random initial weights.

**Evidence to retain.** Keep the keypoint matrix, network shape trace, individual PASS/FAIL lines, and the final automatic score.


### Decode and evaluate the result

Complete the single blank marked TODO 12 to find each heatmap's maximum-value index.
Run the cell to print the keypoint matrix, tensor shapes, and automatic checks.
The remaining decoding and evaluation code is supplied.


In [ ]:
def decode_heatmaps(heatmaps, image_size):
    """Convert (17, H/4, W/4) heatmaps to (17, 3) = [x, y, confidence]."""
    joints, h, w = heatmaps.shape
    keypoints = np.zeros((joints, 3), dtype=np.float32)
    for joint, heatmap in enumerate(heatmaps):
        row, col = np.unravel_index(np.argmax(heatmap), heatmap.shape)
        keypoints[joint] = [(col + .5) * image_size / w, (row + .5) * image_size / h, heatmap[row, col]]
    return keypoints

def print_leaf_output_shapes(model, model_input):
    records, hooks = {}, []
    def make_hook(name):
        def hook(_module, _inputs, output):
            records[name] = tuple(output.shape)
        return hook
    for name in ("stem", "high", "down", "low", "fuse", "head"):
        hooks.append(getattr(model, name).register_forward_hook(make_hook(name)))
    with torch.no_grad():
        heatmaps = model(model_input)
    for hook in hooks:
        hook.remove()
    print()
    print("===== Task 3 evidence: network structure and tensor shapes =====")
    print(model)
    print("input:", tuple(model_input.shape))
    for name in ("stem", "high", "down", "low", "fuse", "head"):
        print(f"{name:>5}:", records[name])
    b, c, _, _ = records["fuse"]
    high_h, high_w = records["high"][-2:]
    print("low_up:", (b, c, high_h, high_w))
    return heatmaps

def evaluate_submission(times, left, right, frame_id, fps, pressure_map, matched_time, fused, heatmaps, keypoints):
    expected_index = int(np.argmin(np.abs(times - frame_id / fps)))
    expected = np.zeros((1, 13, 13), dtype=np.float32)
    for i, (row, col) in enumerate(LEFT_SENSOR_POSITIONS):
        expected[0, row, col] = left[expected_index, i]
        expected[0, row, 12 - col] = right[expected_index, i]
    labels = ["pressure arrays", "nearest timestamp", "mirrored 13×13 map", "four-channel finite input", "17 heatmaps", "17×3 decoded keypoints"]
    checks = [
        times.ndim == 1 and left.shape == right.shape and left.shape[1] == 16,
        np.isclose(matched_time, float(times[expected_index])),
        pressure_map.shape == (1, 13, 13) and np.allclose(pressure_map, expected),
        tuple(fused.shape) == (1, 4, IMAGE_SIZE, IMAGE_SIZE) and bool(torch.isfinite(fused).all()),
        heatmaps.ndim == 3 and heatmaps.shape[0] == 17,
        keypoints.shape == (17, 3) and np.isfinite(keypoints).all(),
    ]
    print()
    print("===== Automatic evaluation =====")
    for label, passed in zip(labels, checks):
        print(f"[{'PASS' if passed else 'FAIL'}] {label}")
    print(f"Automatic score: {sum(checks)}/{len(checks)}")
    return sum(checks), len(checks)

model = FourChannelHRNet().eval()
heatmaps = print_leaf_output_shapes(model, fused)[0].cpu().numpy()
keypoints = decode_heatmaps(heatmaps, IMAGE_SIZE)
print("heatmaps:", heatmaps.shape)
print("keypoints shape:", keypoints.shape)
print("keypoint matrix [joint_id, x, y, confidence]:")
print(np.column_stack([np.arange(17), keypoints]))
evaluate_submission(times, left, right, FRAME_ID, fps, pressure_map, matched_time, fused, heatmaps, keypoints)


## Submission checklist

Fill all 12 blanks, then restart the kernel and run all cells in order.
Submit the notebook with the following outputs saved:

1. Frame time, matched pressure time, timing error, and the complete 13×13 pressure map.
2. The four-channel tensor shape and value ranges.
3. The network structure and tensor shape trace.
4. The keypoint matrix and final PASS/FAIL checks.

Suggested filename: `DISC5105_Week8_DATAFUSION_A1_<YourName>.ipynb`.

The network is randomly initialised, so the keypoints demonstrate the pipeline rather
than accurate pose estimates. No additional written discussion is required.
